# Stage-1 Tokenizer Paper Figures

This notebook is a paper-oriented wrapper for the stage-1 RZSM tokenizer comparison.

It is designed to generate a minimal but review-friendly figure package while the stage-2 experiments are still running:
- **Fig. 2**: stage-1 training/convergence audit
- **Fig. 3**: multi-case georeferenced reconstruction audit
- **Fig. 4**: batch benchmark with paired comparisons


In [ ]:
import sys
from pathlib import Path
from datetime import datetime


def find_gptcast_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(8):
        if (p / 'gptcast').is_dir():
            return p
        if (p / 'GPTCast' / 'gptcast').is_dir():
            return (p / 'GPTCast').resolve()
        p = p.parent
    return start.resolve()


ROOT = find_gptcast_root(Path.cwd())
NOTEBOOK_DIR = ROOT / 'notebooks' / 'rzsm'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print('Using ROOT:', ROOT)
print('Notebook helper dir:', NOTEBOOK_DIR)

from stage1_tokenizer_paper_figures import (
    CaseSpec,
    SearchConfig,
    default_cases,
    make_benchmark_figure,
    make_case_study_figure,
    plot_stage1_training_audit,
    set_seed,
)


## Configuration

Edit the ckpt paths, case list, and output directory here if needed.


In [ ]:
ckpt_old = ROOT / 'models' / 'vae_mae_rzsm.ckpt'
ckpt_new = ROOT / 'models' / 'vae_phuber_rzsm.ckpt'
base_dir = ROOT / 'data' / '0.1' / '1' / 'land_surface'
output_dir = ROOT / 'outputs' / 'stage1_tokenizer_paper'

assert ckpt_old.exists(), f'Missing: {ckpt_old}'
assert ckpt_new.exists(), f'Missing: {ckpt_new}'
assert base_dir.exists(), f'Missing: {base_dir}'

run_map = {
    'old_mae_tokenizer': ROOT / 'logs' / 'train' / 'runs' / '2026-03-13_10-47-01',
    'new_phuber_tokenizer': ROOT / 'logs' / 'train' / 'runs' / '2026-03-13_10-49-49',
}
for label, run_dir in run_map.items():
    assert run_dir.exists(), f'Missing run dir for {label}: {run_dir}'

cases = default_cases()
# Example of a manual override:
# cases = [
#     CaseSpec(name='North China Plain', t0=datetime(2020, 4, 6, 12, 0), lat=36.5, lon=116.5),
#     CaseSpec(name='Middle-Lower Yangtze', t0=datetime(2019, 8, 18, 12, 0), lat=30.5, lon=114.5),
#     CaseSpec(name='Southeast Hills', t0=datetime(2020, 10, 8, 12, 0), lat=27.5, lon=118.0),
# ]

search_cfg = SearchConfig(
    resize=720,
    crop=256,
    max_mask_frac=0.40,
    search_attempts=128,
    jitter_px=110,
    selection_mode='dynamic_near_roi',
    top_candidates_to_show=5,
)

benchmark_crops_per_case = 6

print('Output dir:', output_dir)
print('Cases:')
for case in cases:
    print(f'  - {case.name}: {case.t0:%Y-%m-%d} @ lat={case.lat}, lon={case.lon}')


## Fig. 2: Stage-1 training audit

This section saves convergence curves for the two stage-1 runs. The absolute total loss is intentionally omitted because the training objectives differ.


In [ ]:
set_seed(42)
training_df, training_summary = plot_stage1_training_audit(
    run_map=run_map,
    output_dir=output_dir,
    save_stem='fig2_stage1_training_audit',
)
print(training_summary)
training_df


## Fig. 3: Multi-case georeferenced reconstruction audit

For each case, the helper searches for a locally informative crop near the ROI and then plots the most discriminative lead with longitude/latitude axes.


In [ ]:
set_seed(42)
case_df, case_summary = make_case_study_figure(
    ckpt_old=ckpt_old,
    ckpt_new=ckpt_new,
    cases=cases,
    base_dir=base_dir,
    output_dir=output_dir,
    search=search_cfg,
    save_stem='fig3_stage1_case_studies',
)
print(case_summary)
case_df


## Fig. 4: Batch benchmark and paired comparison

This section benchmarks the top informative crops around each case and exports a CSV/JSON summary for paper tables.


In [ ]:
set_seed(42)
bench_df, bench_summary = make_benchmark_figure(
    ckpt_old=ckpt_old,
    ckpt_new=ckpt_new,
    cases=cases,
    base_dir=base_dir,
    output_dir=output_dir,
    benchmark_crops_per_case=benchmark_crops_per_case,
    search=search_cfg,
    save_stem='fig4_stage1_benchmark',
)
print(bench_summary)
bench_df.head()
